# Native matrix-QPE precision study

Preserve the original seeded8x8 matrix and every precision1..8. Native programs use explicit literal fractional register signatures, allocate zero data, run QPE and retain the dropped data environment. Every native and converted state is compared to an independent complete Circuit. All native resource metrics are measured; source Qiskit arrays are explicitly historical. The original matrix shift is computed but unused in its exponential; this behavior is preserved and documented.


In [ ]:
import math
from pathlib import Path
from typing import Annotated

import numpy as np
import torch
from IPython.display import Image, display

from examples.synthesis.tutorials.technology_demonstrations.QPE.qpe_for_unitary_matrix.overview.physics import (
    HISTORICAL_QISKIT_CX,
    HISTORICAL_QISKIT_DEPTHS,
    PRECISIONS,
    UNITARY,
)
from examples.synthesis.tutorials.technology_demonstrations.QPE.qpe_for_unitary_matrix.overview.workflow import (
    StudyPoint,
    plot_study,
    verify_program,
)
from flagquantum import synth
from flagquantum.synth import (
    Output,
    QArray,
    QBit,
    QNum,
    allocate,
    drop,
    qfunc,
    qpe,
    unitary,
)

rng = np.random.RandomState(1235)
raw = rng.rand(8, 8)
new_matrix = torch.tensor((raw + raw.T) / 2, dtype=torch.float64)
spectrum = torch.linalg.eigvalsh(new_matrix)
maximum = spectrum.abs().max()
unused_shifted_matrix = (new_matrix + maximum) / (2 * maximum)
my_unitary = torch.matrix_exp(2j * math.pi * new_matrix)
torch.testing.assert_close(
    my_unitary, torch.tensor(UNITARY, dtype=torch.complex128), atol=1e-14, rtol=0
)
assert new_matrix.shape == (8, 8) and tuple(range(1, 9)) == PRECISIONS
print("Precisions:", PRECISIONS, "Original unshifted spectrum:", spectrum)
print("The computed shift is unused; actual eigenphases wrap modulo one.")


In [ ]:
target_basis = ("cx", "u3")
print("Native costs use actual operator-exact CX/U3 conversion.")
print("Source Qiskit arrays are historical; no new external transpilation is claimed.")


In [ ]:
@qfunc
def precision_1(phase: Output[Annotated[QNum, (1, False, 1)]]) -> None:
    state: QArray[QBit] = QArray()
    allocate(3, state)
    allocate(phase)
    qpe(lambda: unitary(UNITARY, state), phase)
    drop(state)


@qfunc
def precision_2(phase: Output[Annotated[QNum, (2, False, 2)]]) -> None:
    state: QArray[QBit] = QArray()
    allocate(3, state)
    allocate(phase)
    qpe(lambda: unitary(UNITARY, state), phase)
    drop(state)


@qfunc
def precision_3(phase: Output[Annotated[QNum, (3, False, 3)]]) -> None:
    state: QArray[QBit] = QArray()
    allocate(3, state)
    allocate(phase)
    qpe(lambda: unitary(UNITARY, state), phase)
    drop(state)


@qfunc
def precision_4(phase: Output[Annotated[QNum, (4, False, 4)]]) -> None:
    state: QArray[QBit] = QArray()
    allocate(3, state)
    allocate(phase)
    qpe(lambda: unitary(UNITARY, state), phase)
    drop(state)


@qfunc
def precision_5(phase: Output[Annotated[QNum, (5, False, 5)]]) -> None:
    state: QArray[QBit] = QArray()
    allocate(3, state)
    allocate(phase)
    qpe(lambda: unitary(UNITARY, state), phase)
    drop(state)


@qfunc
def precision_6(phase: Output[Annotated[QNum, (6, False, 6)]]) -> None:
    state: QArray[QBit] = QArray()
    allocate(3, state)
    allocate(phase)
    qpe(lambda: unitary(UNITARY, state), phase)
    drop(state)


@qfunc
def precision_7(phase: Output[Annotated[QNum, (7, False, 7)]]) -> None:
    state: QArray[QBit] = QArray()
    allocate(3, state)
    allocate(phase)
    qpe(lambda: unitary(UNITARY, state), phase)
    drop(state)


@qfunc
def precision_8(phase: Output[Annotated[QNum, (8, False, 8)]]) -> None:
    state: QArray[QBit] = QArray()
    allocate(3, state)
    allocate(phase)
    qpe(lambda: unitary(UNITARY, state), phase)
    drop(state)


entries = (
    precision_1,
    precision_2,
    precision_3,
    precision_4,
    precision_5,
    precision_6,
    precision_7,
    precision_8,
)
points: list[StudyPoint] = []
for precision, entry in zip(PRECISIONS, entries, strict=True):
    program = synth.synthesize(entry, max_workspace=0, memory_limit_bytes=64 * 1024**2)
    point = verify_program(program, precision)
    points.append(point)
    print(point)
print("Native depths:", [p.resources.depth for p in points])
print("Native CX counts:", [p.resources.operation_counts["cx"] for p in points])


In [ ]:
assert HISTORICAL_QISKIT_DEPTHS == (320, 950, 2206, 4706, 9694, 19658, 39574, 79394)
assert HISTORICAL_QISKIT_CX == (162, 484, 1124, 2398, 4938, 10008, 20136, 40378)
print("Historical source Qiskit depths:", HISTORICAL_QISKIT_DEPTHS)
print("Historical source Qiskit CX counts:", HISTORICAL_QISKIT_CX)


## Historical external reference

The original next two code cells contain only commented installation and Qiskit transpilation instructions. They perform no operation in the source Notebook. This native version preserves their purpose as clearly labeled historical reference metadata, without installing another SDK or claiming those counts were regenerated. A controlled current comparison would require its own pinned Qiskit environment and acceptance.


In [ ]:
historical_qiskit_version = "1.0.0"
print("Source optional commented environment:", historical_qiskit_version)


In [ ]:
historical_basis = ("cx", "u")
historical_optimization_level = 3
print(
    "Source commented reference basis:",
    historical_basis,
    "optimization level:",
    historical_optimization_level,
)
assert len(points) == 8 and all(p.resources.n_qubits == p.precision + 3 for p in points)


In [ ]:
plot_path = plot_study(
    tuple(points), Path("artifacts/development/matrix_qpe_precision")
)
display(Image(filename=str(plot_path)))
